# Pure SVD - Proposal Compliant

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
# Keep linear algebra backends single-threaded to reduce noise in timing
# and to improve reproducibility across machines.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd

# Global random seed used by all stochastic steps in this notebook.
np.random.seed(42)

## Preprocessing and evaluation protocol


In [ ]:
from pathlib import Path

# -----------------------------
# 1) Load raw input tables
# -----------------------------
# All baselines share the same CSV inputs so results are comparable.
DATA_DIR = Path(".")
MOVIES_PATH = DATA_DIR / "movies.csv"
RATINGS_PATH = DATA_DIR / "ratings.csv"
USERS_PATH = DATA_DIR / "user_profiles.csv"

movies_df = pd.read_csv(MOVIES_PATH)
ratings_df = pd.read_csv(RATINGS_PATH)
user_profiles_df = pd.read_csv(USERS_PATH)

# -----------------------------
# 2) Standardize schemas
# -----------------------------
# Rename columns to a unified format expected by all methods.
ratings = ratings_df.rename(columns={"id": "user_id", "rate": "rating"}).copy()
movies = movies_df.copy()
users = user_profiles_df.rename(columns={"id": "user_id"}).copy()

# Keep only columns needed in this notebook and remove obvious invalid rows.
ratings = ratings[["user_id", "movie_id", "rating"]].dropna()
movies = movies.drop_duplicates(subset=["id"]).copy()
users = users.drop_duplicates(subset=["user_id"]).copy()

# Keep only user flags that actually exist in the current dataset.
flag_cols = [c for c in ["early_bird", "night_owl", "weekend_tweeter"] if c in users.columns]
if "followers_count" not in users.columns:
    raise ValueError("Expected a followers_count column in user_profiles.csv")

# Ensure optional item metadata columns always exist.
for col in ["genres", "actors", "directors"]:
    if col not in movies.columns:
        movies[col] = ""

# Fill missing metadata to prevent tokenization/feature errors.
movies["genres"] = movies["genres"].fillna("")
movies["actors"] = movies["actors"].fillna("")
movies["directors"] = movies["directors"].fillna("")
movies["year_published"] = pd.to_numeric(movies.get("year_published"), errors="coerce")


def split_tokens(value):
    # Parse pipe-separated fields like "Action|Drama" into token lists.
    if pd.isna(value) or value == "":
        return []
    return [t.strip() for t in str(value).split("|") if t.strip()]


def split_user_id_holdout(ratings_df, test_size=0.20, seed=42):
    # User-based split: each user_id belongs to exactly one split.
    rng = np.random.default_rng(seed)

    user_ids = ratings_df["user_id"].dropna().astype(int).unique().copy()
    rng.shuffle(user_ids)

    n_users = len(user_ids)
    n_train = int(n_users * (1.0 - test_size))

    train_users = user_ids[:n_train]
    test_users = user_ids[n_train:]

    train_df = ratings_df[ratings_df["user_id"].astype(int).isin(train_users)].copy()
    test_df = ratings_df[ratings_df["user_id"].astype(int).isin(test_users)].copy()

    return train_df, test_df, train_users, test_users


train_df, test_df, train_users, test_users = split_user_id_holdout(
    ratings, test_size=0.20, seed=42
)

# Cached list of all genre tokens (useful for consistency checks/extensions).
all_genres = sorted({g for s in movies["genres"].fillna("") for g in split_tokens(s)})
all_item_set = set(movies["id"].dropna().astype(int).tolist())


def build_user_pos_lookup(df):
    # Build dictionary: user_id -> set(positive movie_ids).
    lookup = {}
    for uid, group in df.groupby("user_id"):
        lookup[int(uid)] = set(group["movie_id"].astype(int).tolist())
    return lookup


def merge_user_pos_lookups(*lookups):
    # Merge multiple user->set(item) maps into one map.
    merged = {}
    for lookup in lookups:
        for uid, items in lookup.items():
            if uid not in merged:
                merged[uid] = set()
            merged[uid].update(items)
    return merged


train_user_pos = build_user_pos_lookup(train_df)
test_user_pos = build_user_pos_lookup(test_df)
all_user_pos = merge_user_pos_lookups(train_user_pos, test_user_pos)


def sample_negatives(candidate_pool, exclude_set, n_negatives=20, rng=None):
    # Sample negative candidate items while excluding known positives.
    # With replacement is enabled only when pool is smaller than request.
    rng = rng or np.random.default_rng(42)
    pool = np.array([i for i in candidate_pool if i not in exclude_set], dtype=int)
    if len(pool) == 0:
        return np.array([], dtype=int)
    replace = len(pool) < n_negatives
    return rng.choice(pool, size=n_negatives, replace=replace)


def ranking_metrics_from_rank(rank, k=10):
    # Convert a positive item's rank into ranking metrics.
    # Recall@K is binary per example in this one-positive setup.
    recall = 1.0 if rank <= k else 0.0
    ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0.0
    mrr = 1.0 / rank
    return recall, ndcg, mrr


def rank_positive_among_candidates(scores, positive_mask, rng=None):
    # Tie-safe rank: if multiple candidates share the same score as positive,
    # randomize only within the tie group to avoid order bias.
    rng = rng or np.random.default_rng(42)
    positive_idx = int(np.flatnonzero(positive_mask)[0])
    pos_score = float(scores[positive_idx])

    better = int(np.sum(scores > pos_score))
    tie_indices = np.flatnonzero(scores == pos_score)
    tie_count = int(len(tie_indices))

    if tie_count <= 1:
        return better + 1, tie_count

    shuffled_ties = tie_indices[rng.permutation(tie_count)]
    tie_pos = int(np.where(shuffled_ties == positive_idx)[0][0])
    return better + tie_pos + 1, tie_count


def evaluate_ranker(score_fn, positives_df, candidate_pool, all_user_pos_lookup, n_negatives=20, k=10, seed=42):
    # Evaluate a scoring function in sampled ranking setting:
    # one positive + N negatives per interaction.
    rng = np.random.default_rng(seed)
    rows = []
    tie_cases = 0

    for uid, group in positives_df.groupby("user_id"):
        uid = int(uid)
        known_positives = all_user_pos_lookup.get(uid, set())

        for _, row in group.iterrows():
            pos_item = int(row["movie_id"])
            exclude = set(known_positives)
            exclude.add(pos_item)

            negatives = sample_negatives(candidate_pool, exclude, n_negatives=n_negatives, rng=rng)
            candidate_items = np.array([pos_item] + negatives.tolist(), dtype=int)
            positive_mask = np.zeros(len(candidate_items), dtype=bool)
            positive_mask[0] = True

            # Shuffle candidate order so metric does not depend on [positive + negatives] layout.
            perm = rng.permutation(len(candidate_items))
            candidate_items = candidate_items[perm]
            positive_mask = positive_mask[perm]

            scores = score_fn(uid, candidate_items)
            rank, tie_count = rank_positive_among_candidates(scores, positive_mask, rng=rng)
            if tie_count > 1:
                tie_cases += 1

            recall, ndcg, mrr = ranking_metrics_from_rank(rank, k=k)
            rows.append((recall, ndcg, mrr))

    if not rows:
        return {"Recall@10": np.nan, "NDCG@10": np.nan, "MRR": np.nan, "n_eval": 0, "tie_rate": np.nan}

    arr = np.array(rows, dtype=float)
    n_eval = int(len(rows))
    return {
        "Recall@10": float(arr[:, 0].mean()),
        "NDCG@10": float(arr[:, 1].mean()),
        "MRR": float(arr[:, 2].mean()),
        "n_eval": n_eval,
        "tie_rate": float(tie_cases / n_eval),
    }


print(f"Movies: {movies.shape[0]:,} | Ratings: {ratings.shape[0]:,} | Users: {users.shape[0]:,}")
print(f"User split -> train users: {len(train_users):,}, test users: {len(test_users):,}")
print(f"Split rows -> train: {len(train_df):,}, test: {len(test_df):,}")

Movies: 78,628 | Ratings: 1,172,038 | Users: 482
User split -> train users: 488, test users: 123
Split rows -> train: 934,426, test: 237,612


## Method-specific training and evaluation

In [ ]:
from scipy.sparse.linalg import svds

# -------------------------------------------
# 3) Build PureSVD latent factor model
# -------------------------------------------

def train_pure_svd(train_df, n_factors=64):
    # Build user/item ID lists seen in training interactions.
    train_users = sorted(train_df["user_id"].astype(int).unique().tolist())
    train_items_local = sorted(train_df["movie_id"].astype(int).unique().tolist())

    # Map raw IDs to matrix indices.
    u_map = {u: i for i, u in enumerate(train_users)}
    i_map = {m: j for j, m in enumerate(train_items_local)}

    # Construct dense user-item rating matrix R.
    # Unobserved entries remain zero before mean-filling.
    R = np.zeros((len(train_users), len(train_items_local)), dtype=np.float32)
    for _, row in train_df.iterrows():
        u = int(row["user_id"])
        m = int(row["movie_id"])
        if u in u_map and m in i_map:
            R[u_map[u], i_map[m]] = float(row["rating"])

    # Mean-fill missing values so SVD can operate on a dense matrix.
    global_mean = R[R > 0].mean() if np.any(R > 0) else 0.0
    R_filled = R.copy()
    R_filled[R_filled == 0] = global_mean

    # Choose valid rank k and compute truncated SVD.
    k = min(n_factors, min(R_filled.shape) - 1) if min(R_filled.shape) > 1 else 1
    U, s, Vt = svds(R_filled, k=k)

    # Reconstruct score matrix from latent factors.
    preds = U @ np.diag(s) @ Vt

    return {"u_map": u_map, "i_map": i_map, "preds": preds, "global_mean": float(global_mean)}


# Train PureSVD model on the training split.
svd_model = train_pure_svd(train_df, n_factors=64)


def svd_score_fn(uid, candidate_items):
    # Score candidate items for one user using reconstructed matrix values.
    # Fallback to global mean if user/item was unseen during training.
    u_map = svd_model["u_map"]
    i_map = svd_model["i_map"]
    preds = svd_model["preds"]
    global_mean = svd_model["global_mean"]

    scores = []
    for item in candidate_items:
        if uid in u_map and item in i_map:
            scores.append(float(preds[u_map[uid], i_map[item]]))
        else:
            scores.append(global_mean)
    return np.array(scores, dtype=float)


# -------------------------------------------
# 4) Evaluate on test split
# -------------------------------------------
svd_test_metrics = evaluate_ranker(
    svd_score_fn,
    test_df,
    all_item_set,
    all_user_pos,
    n_negatives=20,
    k=10,
    seed=43,
)

print("Pure SVD")
print("Test:", svd_test_metrics)

Pure SVD
Test: {'Recall@10': 0.476571048600239, 'NDCG@10': 0.21627618106779894, 'MRR': 0.1733230361999986, 'n_eval': 237612, 'tie_rate': 1.0}
